In [4]:
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from analytical_cases import *

# Interactive widgets
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display

# Avoid warnings for divisions by zero (e.g., r=0)
np.seterr(divide='ignore', invalid='ignore')

{'divide': 'ignore', 'over': 'warn', 'under': 'ignore', 'invalid': 'ignore'}

In [5]:
def interactive_strain_plot(filename="output/fields.vtu",
                            strain_field_name=None,
                            analytical_case="lame",
                            tol=1e-5):
    """
    Interactive radial strain plot with dropdown menu for Z-level selection.

    Parameters:
        filename (str): Path to the .vtu file.
        strain_field_name (str): Name of the strain field to use. If None, the first detected is used.
        analytical_case (str): Which analytical case to compare against.
        tol (float): Tolerance for selecting Z-level cells.
    """
    print(f"Loading data from: {filename}")
    grid = pv.read(filename)

    # Auto-detect strain field if not specified
    if strain_field_name is None:
        available_strains = [k for k in grid.cell_data if "Strain" in k]
        if not available_strains:
            print("[ERROR] No 'Strain' field found in the file.")
            return
        strain_field_name = available_strains[0]
        print(f"[INFO] Automatically selected strain field: {strain_field_name}")

    # Extract relevant data once for efficiency
    centers = grid.cell_centers().points
    e_tensor_full = grid.cell_data[strain_field_name].reshape((-1, 3, 3))

    # Identify available Z levels for dropdown selection
    unique_z_coords = np.unique(np.round(centers[:, 2], 6))
    print(f"Found {len(unique_z_coords)} unique Z levels.")

    # Print all available Z coordinates
    print("\nAvailable Z levels for slicing:")
    for z in unique_z_coords:
        print(f"  → z = {z}")

    # Define plotting function based on selected Z level
    def plot_strain_at_z(z_level):
        mask = np.abs(centers[:, 2] - z_level) < tol
        if not np.any(mask):
            print(f"[WARNING] No cells found at z = {z_level} ± {tol}")
            return

        e_tensor = e_tensor_full[mask]
        x, y = centers[mask, 0], centers[mask, 1]

        # Convert to cylindrical coordinates
        r = np.sqrt(x**2 + y**2)
        theta = np.arctan2(y, x)

        c, s = np.cos(theta), np.sin(theta)
        c2, s2, sc = c**2, s**2, s*c
        e_xx, e_yy, e_xy, e_zz = e_tensor[:,0,0], e_tensor[:,1,1], e_tensor[:,0,1], e_tensor[:,2,2]

        # Cylindrical components
        eps_rr = e_xx * c2 + e_yy * s2 + 2 * e_xy * sc
        eps_tt = e_xx * s2 + e_yy * c2 - 2 * e_xy * sc
        eps_zz = e_zz

        # Group and average by radial coordinate
        df = pd.DataFrame({"r": np.round(r, 6), "eps_rr": eps_rr, "eps_tt": eps_tt, "eps_zz": eps_zz})
        grouped = df.groupby("r").mean().reset_index()
        r_avg = grouped["r"].values

        # Analytical solution
        if analytical_case is None:
            eps_tt_an, eps_rr_an = np.zeros_like(r_avg), np.zeros_like(r_avg)
        else:
            u_r_an = radial_displacement(r_avg)
            eps_tt_an = u_r_an / r_avg
            eps_rr_an = radial_strain(r_avg)

        # Plotting
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.plot(r_avg, grouped["eps_rr"], 'o', c='darkgreen', markersize=5, label="ε_rr (Radial) (FEM)")
        ax.plot(r_avg, grouped["eps_tt"], 's', c='crimson', markersize=5, label="ε_tt (Hoop) (FEM)")
        ax.plot(r_avg, grouped["eps_zz"], 'x', c='lightblue', markersize=6, label="ε_zz (Axial) (FEM)")

        ax.plot(r_avg, eps_rr_an, '--', c='darkgreen', lw=2, label="ε_rr (Analytical)")
        ax.plot(r_avg, eps_tt_an, '--', c='crimson', lw=2, label="ε_tt (Analytical)")

        ax.set_xlabel("Radius r (m)", fontsize=12)
        ax.set_ylabel("Strain ε (-)", fontsize=12)
        ax.set_title(f"Strain components vs. radius at z = {z_level}", fontsize=14)
        ax.grid(True, linestyle=':')
        ax.legend()
        plt.tight_layout()
        plt.show()

    # Dropdown widget for selecting Z level
    interact(
        plot_strain_at_z,
        z_level=widgets.Dropdown(
            options=unique_z_coords,
            value=unique_z_coords[len(unique_z_coords) // 2],
            description='Select Z-level:',
            style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )
    )

# --- Usage ---
# Run this in a Jupyter Notebook cell to launch the interactive tool
if __name__ == "__main__":
    interactive_strain_plot(
        filename="output/fields.vtu",
        analytical_case="lame"
    )


Loading data from: output/fields.vtu
[INFO] Automatically selected strain field: Strain_steel
Found 21 unique Z levels.

Available Z levels for slicing:
  → z = 0.238095
  → z = 0.714286
  → z = 1.190476
  → z = 1.666667
  → z = 2.142857
  → z = 2.619048
  → z = 3.095238
  → z = 3.571429
  → z = 4.047619
  → z = 4.52381
  → z = 5.0
  → z = 5.47619
  → z = 5.952381
  → z = 6.428571
  → z = 6.904762
  → z = 7.380952
  → z = 7.857143
  → z = 8.333333
  → z = 8.809524
  → z = 9.285714
  → z = 9.761905


interactive(children=(Dropdown(description='Select Z-level:', index=10, layout=Layout(width='50%'), options=(0…

In [6]:
def interactive_stress_plot(filename="output/fields.vtu",
                            stress_field_name=None,
                            analytical_case="lame",
                            tol=1e-5):
    """
    Interactive radial stress plot with dropdown menu for Z-level selection.

    Parameters:
        filename (str): Path to the .vtu file.
        stress_field_name (str): Name of the stress field to use. If None, the first detected is used.
        analytical_case (str): Which analytical case to compare against.
        tol (float): Tolerance for selecting Z-level cells.
    """
    print(f"Loading data from: {filename}")
    grid = pv.read(filename)

    if stress_field_name is None:
        available_stresses = [k for k in grid.cell_data if "Stress" in k]
        if not available_stresses:
            print("[ERROR] No 'Stress' field found in the file.")
            return
        stress_field_name = available_stresses[0]
        print(f"[INFO] Automatically selected stress field: {stress_field_name}")

    centers = grid.cell_centers().points
    s_tensor_full = grid.cell_data[stress_field_name].reshape((-1, 3, 3))

    unique_z_coords = np.unique(np.round(centers[:, 2], 6))
    print(f"\nAvailable Z levels for slicing:")
    for z in unique_z_coords:
        print(f"  → z = {z}")

    def plot_stress_at_z(z_level):
        mask = np.abs(centers[:, 2] - z_level) < tol
        if not np.any(mask):
            print(f"[WARNING] No cells found at z = {z_level} ± {tol}")
            return

        s_tensor = s_tensor_full[mask]
        x, y = centers[mask, 0], centers[mask, 1]

        r = np.sqrt(x**2 + y**2)
        theta = np.arctan2(y, x)

        c, s = np.cos(theta), np.sin(theta)
        c2, s2, sc = c**2, s**2, s*c
        s_xx, s_yy, s_xy, s_zz = s_tensor[:,0,0], s_tensor[:,1,1], s_tensor[:,0,1], s_tensor[:,2,2]

        σ_rr = s_xx * c2 + s_yy * s2 + 2 * s_xy * sc
        σ_tt = s_xx * s2 + s_yy * c2 - 2 * s_xy * sc
        σ_zz = s_zz

        df = pd.DataFrame({"r": np.round(r, 6), "σ_rr": σ_rr, "σ_tt": σ_tt, "σ_zz": σ_zz})
        grouped = df.groupby("r").mean().reset_index()
        r_avg = grouped["r"].values

        # Soluzione analitica
        if analytical_case == "lame":
            σ_rr_an, σ_tt_an, σ_zz_an, _ = lame_solutions(r_avg)
        elif analytical_case == "thermal":
            σ_rr_an, σ_tt_an, σ_zz_an = thermal_stresses(r_avg)
        else:
            σ_rr_an = σ_tt_an = σ_zz_an = np.zeros_like(r_avg)

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.plot(r_avg, grouped["σ_rr"], 'o', c='purple', markersize=5, label="σ_rr (Radial) (FEM)")
        ax.plot(r_avg, grouped["σ_tt"], 's', c='orange', markersize=5, label="σ_tt (Hoop) (FEM)")
        ax.plot(r_avg, grouped["σ_zz"], '^', c='blue', markersize=5, label="σ_zz (Axial) (FEM)")

        ax.plot(r_avg, σ_rr_an, '--', c='purple', lw=2, label="σ_rr (Analytical)")
        ax.plot(r_avg, σ_tt_an, '--', c='orange', lw=2, label="σ_tt (Analytical)")
        ax.plot(r_avg, σ_zz_an, '--', c='blue', lw=2, label="σ_zz (Analytical)")

        ax.set_xlabel("Radius r (m)", fontsize=12)
        ax.set_ylabel("Stress σ (Pa)", fontsize=12)
        ax.set_title(f"Stress components vs. radius at z = {z_level}", fontsize=14)
        ax.grid(True, linestyle=':')
        ax.legend()
        plt.tight_layout()
        plt.show()

    interact(
        plot_stress_at_z,
        z_level=widgets.Dropdown(
            options=unique_z_coords,
            value=unique_z_coords[len(unique_z_coords) // 2],
            description='Select Z-level:',
            style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )
    )

# --- Usage ---
# Run this in a Jupyter Notebook cell to launch the interactive tool
if __name__ == "__main__":
    interactive_stress_plot(
        filename="output/fields.vtu",
        analytical_case="lame"
    )


Loading data from: output/fields.vtu
[INFO] Automatically selected stress field: Stress_steel

Available Z levels for slicing:
  → z = 0.238095
  → z = 0.714286
  → z = 1.190476
  → z = 1.666667
  → z = 2.142857
  → z = 2.619048
  → z = 3.095238
  → z = 3.571429
  → z = 4.047619
  → z = 4.52381
  → z = 5.0
  → z = 5.47619
  → z = 5.952381
  → z = 6.428571
  → z = 6.904762
  → z = 7.380952
  → z = 7.857143
  → z = 8.333333
  → z = 8.809524
  → z = 9.285714
  → z = 9.761905


interactive(children=(Dropdown(description='Select Z-level:', index=10, layout=Layout(width='50%'), options=(0…